# State, Sensors, and Run Data
Companion notebook for the **State, Sensors, and Run Data** section of Chapter 7.

Not all inputs are the same. **Live signals** (a webhook, a fresh tool result) belong in a `sensors`/`observations` area, not automatically in long-term memory. Step-by-step execution details belong in `run_data`. Only selected, validated items move into memory later.

In [1]:
from typing import TypedDict


class TicketEvent(TypedDict):
    ticket_id: str
    status: str
    received_at: str


class ToolCall(TypedDict):
    tool: str
    status: str


class RunData(TypedDict):
    current_step: str
    tool_calls: list[ToolCall]


class MemoryCandidate(TypedDict):
    key: str
    value: object
    source: str

The book's example: a live sensor reading, run-in-progress data, and a *candidate* for memory that hasn't been promoted yet:

In [2]:
import json

sensor_state = {
    "sensors": {
        "latest_ticket_event": {
            "ticket_id": "T-102",
            "status": "customer_replied",
            "received_at": "2026-04-27T10:15:00Z",
        }
    },
    "run_data": {
        "current_step": "decide_next_action",
        "tool_calls": [
            {"tool": "ticket_api", "status": "success"},
        ],
    },
    "memory_candidates": [
        {"key": "customer_sent_logs", "value": True, "source": "ticket_event"},
    ],
}
print(json.dumps(sensor_state, indent=2))

{
  "sensors": {
    "latest_ticket_event": {
      "ticket_id": "T-102",
      "status": "customer_replied",
      "received_at": "2026-04-27T10:15:00Z"
    }
  },
  "run_data": {
    "current_step": "decide_next_action",
    "tool_calls": [
      {
        "tool": "ticket_api",
        "status": "success"
      }
    ]
  },
  "memory_candidates": [
    {
      "key": "customer_sent_logs",
      "value": true,
      "source": "ticket_event"
    }
  ]
}


Keep raw tool output *and* a cleaned summary, in separate fields. Raw output is useful for audit, replay, and debugging; later nodes usually need the short summary, not the full payload:

In [3]:
tool_output = {
    "tool_output_raw": {
        "inventory_api": {
            "sku": "A12",
            "warehouse": "west",
            "stock": 0,
            "checked_at": "2026-04-27T10:20:00Z",
        }
    },
    "tool_output_summary": {
        "inventory_status": "out_of_stock",
    },
}
print(json.dumps(tool_output, indent=2))

# A later node reads the short summary, not the raw payload:
assert tool_output["tool_output_summary"]["inventory_status"] == "out_of_stock"

{
  "tool_output_raw": {
    "inventory_api": {
      "sku": "A12",
      "warehouse": "west",
      "stock": 0,
      "checked_at": "2026-04-27T10:20:00Z"
    }
  },
  "tool_output_summary": {
    "inventory_status": "out_of_stock"
  }
}
